# 04 Sentence-Transformer Training

This notebook is self-contained. The graph remains explainable and corpus-based; the neural model is used separately for semantic poem retrieval.

The deep learning requirement is handled here with a small sentence-transformer fine-tuning task. This model does not create the symbol-emotion graph. Instead, it supports the application feature where a user pastes a poem and receives semantically similar poems from the corpus.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models' / 'sentence_transformer_poetry'
POEMS_CLEAN_PATH = PROCESSED_DIR / 'poems_clean.csv'
SYMBOLS_PATH = PROCESSED_DIR / 'extracted_symbols.csv'
EMOTIONS_PATH = PROCESSED_DIR / 'extracted_emotions.csv'
RELATIONS_PATH = PROCESSED_DIR / 'symbol_emotion_edges.csv'
SIMILARITY_PAIRS_PATH = PROCESSED_DIR / 'poem_similarity_pairs.csv'
EMBEDDINGS_PATH = PROCESSED_DIR / 'poem_embeddings.npy'
EMBEDDING_METADATA_PATH = PROCESSED_DIR / 'poem_embedding_metadata.csv'
BASE_SENTENCE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

c:\Users\Lenovo X1 Carbon\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

Weak positive pairs come from shared author, tag, emotion, symbol-emotion relation, or at least two shared symbols. Negative pairs avoid those overlaps.

This is called weak supervision because the labels are generated from corpus metadata and extracted graph features rather than manual annotation. The goal is not to create a perfect similarity model, but to adapt a lightweight pretrained model toward poetry retrieval.

In [2]:
def generate_similarity_pairs(poems_df, symbols_df, emotions_df, relations_df, max_positive=3000, max_negative=3000):
    features = {poem_id: {'symbols': set(), 'emotions': set(), 'relations': set(), 'tags': set(), 'author': ''} for poem_id in poems_df['poem_id']}
    for poem in poems_df.itertuples(index=False):
        features[poem.poem_id]['tags'] = {tag.strip().lower() for tag in str(poem.tags).replace(';', ',').split(',') if tag.strip()}
        features[poem.poem_id]['author'] = str(poem.author).lower()
    for row in symbols_df.itertuples(index=False):
        features[row.poem_id]['symbols'].add(row.symbol)
    for row in emotions_df.itertuples(index=False):
        features[row.poem_id]['emotions'].add(row.emotion_category)
    for row in relations_df.itertuples(index=False):
        features[row.poem_id]['relations'].add((row.source_symbol, row.target_emotion))
    text_by_id = poems_df.set_index('poem_id')['clean_text'].to_dict()
    ids = list(features.keys())
    positives = []
    negatives = []
    attempts = 0
    while len(positives) < max_positive and attempts < max_positive * 30:
        a, b = random.sample(ids, 2)
        fa, fb = features[a], features[b]
        reasons = []
        if fa['author'] and fa['author'] == fb['author']:
            reasons.append('same_author')
        if fa['tags'] & fb['tags']:
            reasons.append('shared_tag')
        if fa['emotions'] & fb['emotions']:
            reasons.append('shared_emotion')
        if fa['relations'] & fb['relations']:
            reasons.append('shared_relation')
        if len(fa['symbols'] & fb['symbols']) >= 2:
            reasons.append('shared_symbols')
        if reasons:
            positives.append({'text_a': text_by_id[a], 'text_b': text_by_id[b], 'label': 1.0, 'reason': ','.join(reasons)})
        attempts += 1
    attempts = 0
    while len(negatives) < max_negative and attempts < max_negative * 50:
        a, b = random.sample(ids, 2)
        fa, fb = features[a], features[b]
        unrelated = fa['author'] != fb['author'] and not fa['tags'] & fb['tags'] and not fa['emotions'] & fb['emotions'] and not fa['symbols'] & fb['symbols']
        if unrelated:
            negatives.append({'text_a': text_by_id[a], 'text_b': text_by_id[b], 'label': 0.0, 'reason': 'random_unrelated'})
        attempts += 1
    pairs = pd.DataFrame(positives + negatives).sample(frac=1, random_state=42).reset_index(drop=True)
    pairs.to_csv(SIMILARITY_PAIRS_PATH, index=False)
    return pairs

def train_sentence_transformer(pairs_df, epochs=1, batch_size=8):
    model = SentenceTransformer(BASE_SENTENCE_MODEL)
    model.max_seq_length = 128
    examples = [InputExample(texts=[row.text_a, row.text_b], label=float(row.label)) for row in pairs_df.itertuples(index=False)]
    loader = DataLoader(examples, shuffle=True, batch_size=batch_size)
    loss = losses.CosineSimilarityLoss(model)
    model.fit(train_objectives=[(loader, loss)], epochs=epochs, show_progress_bar=True)
    model.save(str(MODEL_DIR))
    return model

def encode_poems(model, poems_df):
    embeddings = model.encode(poems_df['clean_text'].fillna('').tolist(), batch_size=8, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    np.save(EMBEDDINGS_PATH, embeddings)
    poems_df[['poem_id', 'title', 'author', 'poem_text']].to_csv(EMBEDDING_METADATA_PATH, index=False)
    return embeddings

## Create Pairs, Train, and Save Embeddings

The model is CPU-friendly: MiniLM, maximum sequence length 128, batch size 8, and one training epoch. After training, every poem is encoded once and saved as an embedding. The app reuses these saved embeddings so it does not need to recompute them every time a user searches.

In [3]:
poems_df = pd.read_csv(POEMS_CLEAN_PATH)
symbols_df = pd.read_csv(SYMBOLS_PATH)
emotions_df = pd.read_csv(EMOTIONS_PATH)
relations_df = pd.read_csv(RELATIONS_PATH)
pairs_df = generate_similarity_pairs(poems_df, symbols_df, emotions_df, relations_df, max_positive=3000, max_negative=3000)
pairs_df.head()

,text_a,text_b,label,reason
0,Aiee! It is the ceremony of the first blades o...,Not that I always struck the proper mean Of wh...,1.0,shared_emotion
1,Two guys sucking each other in the steam room ...,"Toward evening, as the light failed and the pe...",0.0,random_unrelated
2,Between the bridge and the river he falls thro...,My hatwas run overby a trolleyyesterday.This m...,1.0,shared_emotion
3,pulse light starts there starts getting smalle...,"My father in the aluminum stern, cursing anoth...",1.0,shared_emotion
4,"Though there's no such thing as a ""self,"" I mi...",is a fairy-tale queendom with monsters whom I ...,0.0,random_unrelated


In [4]:
model = train_sentence_transformer(pairs_df, epochs=1, batch_size=8)
embeddings = encode_poems(model, poems_df)
embeddings.shape

c:\Users\Lenovo X1 Carbon\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.276300


Batches: 100%|██████████| 1720/1720 [08:01<00:00,  3.57it/s]


(13753, 384)